In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import pathlib
import torch
import pandas

from sklearn.metrics import f1_score


/local/home/fbarkmann/project/sae-for-scFMs/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cell_map = {2: 'alpha',
 3: 'beta',
 5: 'ductal',
 0: 'acinar',
 4: 'delta',
 8: 'gamma',
 1: 'activated',
 6: 'endothelial',
 11: 'quiescent',
 9: 'macrophage',
 10: 'mast',
 7: 'epsilon',
 12: 'schwann',
 13: 'T-cell'}

In [3]:
def normalize_batch_name(batch_name: str) -> str:
    return "inDrop" if str(batch_name).startswith("inDrop") else str(batch_name)

In [4]:
labels = pd.read_csv("../data/embeddings_pancreas/pancreas_labels.csv")
cell_type = labels["celltype"].astype(str)
batch = labels["tech"].astype(str).map(normalize_batch_name)

In [ ]:
import numpy as np
import re

embeddings_root = pathlib.Path("../data/embeddings_pancreas")
seed_pattern = re.compile(r"seed(\d+)")

# Each method maps to (directory, filename pattern).
method_to_files = {
    "scgpt (zero-shot)": (embeddings_root / "scgpt_zeroshot", "original_seed*.npy"),
    "scgpt (zero-shot, steered)": (
        embeddings_root / "scgpt_zeroshot_steered",
        "AMI_n20_seed*_clamp-2.0.npy",
    ),
    "scgpt (fine-tuned)": (embeddings_root / "scgpt_finetuned", "original_seed*.npy"),
    "scgpt (fine-tuned, steered)": (embeddings_root / "scgpt_finetuned_steered", "AMI_n50_seed*_clamp-2.0.npy"),

    "geneformer (fine-tuned)": (embeddings_root / "geneformer_finetuned", "original_seed*.npy"),
    "geneformer (fine-tuned, steered)": (
        embeddings_root / "geneformer_finetuned_steered",
        "AMI_n50_seed*_clamp-2.0.npy",
    ),
}

embeddings = {}
for method_name, (method_path, filename_pattern) in method_to_files.items():
    seed_to_embedding = {}

    for embedding_path in sorted(method_path.glob(filename_pattern)):
        seed_match = seed_pattern.search(embedding_path.stem)
        if seed_match is None:
            continue

        seed = int(seed_match.group(1))
        seed_to_embedding[seed] = np.load(embedding_path)

    ordered_seeds = sorted(seed_to_embedding)
    if len(ordered_seeds) != 3:
        raise ValueError(
            f"Expected exactly 3 seeds for {method_name} in {method_path} "
            f"with pattern {filename_pattern}, found {ordered_seeds}"
        )

    embeddings[method_name] = [seed_to_embedding[seed] for seed in ordered_seeds]
    print(f"Loaded {method_name}: seeds={ordered_seeds}")

Loaded scgpt (zero-shot): seeds=[42, 43, 44]
Loaded scgpt (zero-shot, steered): seeds=[42, 43, 44]
Loaded scgpt (fine-tuned): seeds=[42, 43, 44]
Loaded scgpt (fine-tuned, steered): seeds=[42, 43, 44]
Loaded geneformer (fine-tuned): seeds=[42, 43, 44]
Loaded geneformer (fine-tuned, steered): seeds=[42, 43, 44]


In [25]:
{method: len(seed_embeddings) for method, seed_embeddings in embeddings.items()}

{'scgpt (zero-shot)': 3,
 'scgpt (zero-shot, steered)': 3,
 'scgpt (fine-tuned)': 3,
 'scgpt (fine-tuned, steered)': 3,
 'geneformer (fine-tuned)': 3,
 'geneformer (fine-tuned, steered)': 3}

In [35]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np

unique_batches = batch.unique()
results = {}
results_by_seed = {}

y = cell_type.values
b = batch.values

for emb_name, emb_list in embeddings.items():
    seed_metrics = []

    for emb in emb_list:
        X = emb
        batch_metrics = {}
        for batch_val in unique_batches:
            train_idx = np.where(b != batch_val)[0]
            test_idx = np.where(b == batch_val)[0]

            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            knn = KNeighborsClassifier(n_neighbors=5)
            knn.fit(X_train, y_train)
            y_pred = knn.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            weighted_f1 = f1_score(
                y_test,
                y_pred,
                average="weighted",
                labels=np.unique(y_test),
            )
            macro_f1 = f1_score(
                y_test,
                y_pred,
                average="macro",
                labels=np.unique(y_test),
            )
            batch_metrics[batch_val] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
            }

        seed_metrics.append(batch_metrics)

    results_by_seed[emb_name] = seed_metrics

    aggregated_metrics = {}
    for batch_val in unique_batches:
        acc_values = [seed_result[batch_val]["accuracy"] for seed_result in seed_metrics]
        f1_values = [seed_result[batch_val]["macro_f1"] for seed_result in seed_metrics]
        weighted_f1_values = [seed_result[batch_val]["weighted_f1"] for seed_result in seed_metrics]
        aggregated_metrics[batch_val] = {
            "accuracy": float(np.mean(acc_values)),
            "macro_f1": float(np.mean(f1_values)),
            "weighted_f1": float(np.mean(weighted_f1_values)),
            "accuracy_std": float(np.std(acc_values)),
            "macro_f1_std": float(np.std(f1_values)),
            "weighted_f1_std": float(np.std(weighted_f1_values)),
        }

    results[emb_name] = aggregated_metrics

# Print aggregated mean +/- std across seeds.
for emb_name, batch_accs in results.items():
    print(
        f"Embedding: {emb_name} "
        f"(mean +/- std over {len(results_by_seed[emb_name])} seed(s))"
    )
    for batch_val, metrics in batch_accs.items():
        print(
            f"  Tested on batch {batch_val}, "
            f"Test accuracy: {metrics['accuracy']:.3f} +/- {metrics['accuracy_std']:.3f}, "
            f"Weighted F1: {metrics['macro_f1']:.3f} +/- {metrics['macro_f1_std']:.3f}"
        )

Embedding: scgpt (zero-shot) (mean +/- std over 3 seed(s))
  Tested on batch celseq, Test accuracy: 0.812 +/- 0.005, Weighted F1: 0.534 +/- 0.016
  Tested on batch celseq2, Test accuracy: 0.805 +/- 0.010, Weighted F1: 0.597 +/- 0.009
  Tested on batch fluidigmc1, Test accuracy: 0.848 +/- 0.006, Weighted F1: 0.453 +/- 0.032
  Tested on batch smartseq2, Test accuracy: 0.808 +/- 0.004, Weighted F1: 0.613 +/- 0.003
  Tested on batch inDrop, Test accuracy: 0.770 +/- 0.001, Weighted F1: 0.373 +/- 0.001
  Tested on batch smarter, Test accuracy: 0.866 +/- 0.000, Weighted F1: 0.563 +/- 0.011
Embedding: scgpt (zero-shot, steered) (mean +/- std over 3 seed(s))
  Tested on batch celseq, Test accuracy: 0.846 +/- 0.011, Weighted F1: 0.513 +/- 0.047
  Tested on batch celseq2, Test accuracy: 0.862 +/- 0.006, Weighted F1: 0.602 +/- 0.002
  Tested on batch fluidigmc1, Test accuracy: 0.906 +/- 0.003, Weighted F1: 0.618 +/- 0.039
  Tested on batch smartseq2, Test accuracy: 0.843 +/- 0.000, Weighted F1: 0.

In [38]:
# Build a transposed LaTeX table for all pancreas embedding methods.
metric_key = "accuracy"  # Change to "macro_f1" if you want the Weighted F1 table.
metric_label = "Accuracy"

metric_by_method = {}
for method_name, batch_metrics in results.items():
    pretty_name = method_name
    metric_by_method[pretty_name] = pd.Series(
        {batch_name: metrics[metric_key] for batch_name, metrics in batch_metrics.items()}
    )

metric_df = pd.DataFrame(metric_by_method)
metric_df.index.name = "batch"
metric_df = metric_df.reset_index()
metric_df["batch"] = metric_df["batch"].map(normalize_batch_name)
metric_df = metric_df.groupby("batch", as_index=True).mean(numeric_only=True).sort_index()

# Add deltas as steered - non-steered for each comparable pair.
delta_pairs = [
    ("scgpt (zero-shot, steered)", "scgpt (zero-shot)", "$\\Delta$ scgpt (zero-shot)"),
    ("scgpt (fine-tuned, steered)", "scgpt (fine-tuned)", "$\\Delta$ scgpt (fine-tuned)"),
    (
        "geneformer (fine-tuned, steered)",
        "geneformer (fine-tuned)",
        "$\\Delta$ geneformer (fine-tuned)",
    ),
]
for steered_name, base_name, delta_name in delta_pairs:
    if {steered_name, base_name}.issubset(metric_df.columns):
        metric_df[delta_name] = metric_df[steered_name] - metric_df[base_name]

mean_row = pd.DataFrame(metric_df.mean(numeric_only=True)).T
mean_row.index = ["Mean"]
latex_df = pd.concat([metric_df, mean_row]).T

latex_table = latex_df.to_latex(
    index=True,
    float_format="%.3f",
    caption=(
        f"Transposed cross-batch KNN {metric_label} comparison across pancreas "
        "embedding methods (mean over 3 seeds)."
    ),
    label="tab:pancreas_embedding_methods_transposed",
)

print(latex_table)

\begin{table}
\caption{Transposed cross-batch KNN Accuracy comparison across pancreas embedding methods (mean over 3 seeds).}
\label{tab:pancreas_embedding_methods_transposed}
\begin{tabular}{lrrrrrrr}
\toprule
 & celseq & celseq2 & fluidigmc1 & inDrop & smarter & smartseq2 & Mean \\
\midrule
scgpt (zero-shot) & 0.812 & 0.805 & 0.848 & 0.770 & 0.866 & 0.808 & 0.818 \\
scgpt (zero-shot, steered) & 0.846 & 0.862 & 0.906 & 0.796 & 0.915 & 0.843 & 0.861 \\
scgpt (fine-tuned) & 0.877 & 0.934 & 0.915 & 0.884 & 0.930 & 0.891 & 0.905 \\
scgpt (fine-tuned, steered) & 0.902 & 0.960 & 0.960 & 0.909 & 0.993 & 0.931 & 0.942 \\
geneformer (fine-tuned) & 0.395 & 0.668 & 0.830 & 0.237 & 0.306 & 0.554 & 0.498 \\
geneformer (fine-tuned, steered) & 0.395 & 0.666 & 0.833 & 0.221 & 0.330 & 0.562 & 0.501 \\
$\Delta$ scgpt (zero-shot) & 0.034 & 0.057 & 0.058 & 0.026 & 0.049 & 0.035 & 0.043 \\
$\Delta$ scgpt (fine-tuned) & 0.025 & 0.026 & 0.045 & 0.025 & 0.063 & 0.041 & 0.037 \\
$\Delta$ geneformer (fine-tune

In [6]:
import numpy as np
import re

embeddings_root = pathlib.Path("../data/embeddings_lung")
seed_pattern = re.compile(r"seed(\d+)")

# Each method maps to (directory, filename pattern).
method_to_files = {
    "scgpt (zero-shot)": (embeddings_root / "zeroshot", "original_seed*.npy"),
    "scgpt (zero-shot, steered)": (embeddings_root / "zeroshot_steered", "AMI_n50_seed*_clamp-2.0.npy"),
    "scgpt (fine-tuned)": (embeddings_root / "finetuned", "original_seed*.npy"),
    "scgpt (fine-tuned, steered)": (embeddings_root / "finetuned_steered", "AMI_n50_seed*_clamp-2.0.npy"),
}

embeddings = {}
for method_name, (method_path, filename_pattern) in method_to_files.items():
    seed_to_embedding = {}

    for embedding_path in sorted(method_path.glob(filename_pattern)):
        seed_match = seed_pattern.search(embedding_path.stem)
        if seed_match is None:
            continue

        seed = int(seed_match.group(1))
        seed_to_embedding[seed] = np.load(embedding_path)

    ordered_seeds = sorted(seed_to_embedding)
    if len(ordered_seeds) != 3:
        raise ValueError(
            f"Expected exactly 3 seeds for {method_name} in {method_path} "
            f"with pattern {filename_pattern}, found {ordered_seeds}"
        )

    embeddings[method_name] = [seed_to_embedding[seed] for seed in ordered_seeds]
    print(f"Loaded {method_name}: seeds={ordered_seeds}")

Loaded scgpt (zero-shot): seeds=[42, 43, 44]
Loaded scgpt (zero-shot, steered): seeds=[42, 43, 44]
Loaded scgpt (fine-tuned): seeds=[42, 43, 44]
Loaded scgpt (fine-tuned, steered): seeds=[42, 43, 44]


In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np

results = {}
results_by_seed = {}
lung_labels = pd.read_csv("../data/embeddings_lung/lung_labels.csv")
y = lung_labels["cell_type"].astype(str)
b = lung_labels["dataset"].astype(str)
unique_batches = b.unique()

for emb_name, emb_list in embeddings.items():
    seed_metrics = []

    for emb in emb_list:
        X = emb
        batch_metrics = {}
        for batch_val in unique_batches:
            train_idx = np.where(b != batch_val)[0]
            test_idx = np.where(b == batch_val)[0]

            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            knn = KNeighborsClassifier(n_neighbors=5)
            knn.fit(X_train, y_train)
            y_pred = knn.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            weighted_f1 = f1_score(
                y_test,
                y_pred,
                average="weighted",
                labels=np.unique(y_test),
            )
            macro_f1 = f1_score(
                y_test,
                y_pred,
                average="macro",
                labels=np.unique(y_test),
            )
            batch_metrics[batch_val] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
            }

        seed_metrics.append(batch_metrics)

    results_by_seed[emb_name] = seed_metrics

    aggregated_metrics = {}
    for batch_val in unique_batches:
        acc_values = [seed_result[batch_val]["accuracy"] for seed_result in seed_metrics]
        f1_values = [seed_result[batch_val]["macro_f1"] for seed_result in seed_metrics]
        weighted_f1_values = [seed_result[batch_val]["weighted_f1"] for seed_result in seed_metrics]
        aggregated_metrics[batch_val] = {
            "accuracy": float(np.mean(acc_values)),
            "macro_f1": float(np.mean(f1_values)),
            "weighted_f1": float(np.mean(weighted_f1_values)),
            "accuracy_std": float(np.std(acc_values)),
            "macro_f1_std": float(np.std(f1_values)),
            "weighted_f1_std": float(np.std(weighted_f1_values)),
        }

    results[emb_name] = aggregated_metrics

# Print aggregated mean +/- std across seeds.
for emb_name, batch_accs in results.items():
    print(
        f"Embedding: {emb_name} "
        f"(mean +/- std over {len(results_by_seed[emb_name])} seed(s))"
    )
    for batch_val, metrics in batch_accs.items():
        print(
            f"  Tested on batch {batch_val}, "
            f"Test accuracy: {metrics['accuracy']:.3f} +/- {metrics['accuracy_std']:.3f}, "
            f"Weighted F1: {metrics['macro_f1']:.3f} +/- {metrics['macro_f1_std']:.3f}"
        )

Embedding: scgpt (zero-shot) (mean +/- std over 3 seed(s))
  Tested on batch Dropseq_transplant, Test accuracy: 0.244 +/- 0.002, Weighted F1: 0.212 +/- 0.002
  Tested on batch 10x_Biopsy, Test accuracy: 0.525 +/- 0.003, Weighted F1: 0.442 +/- 0.004
  Tested on batch 10x_Transplant, Test accuracy: 0.701 +/- 0.001, Weighted F1: 0.489 +/- 0.002
Embedding: scgpt (zero-shot, steered) (mean +/- std over 3 seed(s))
  Tested on batch Dropseq_transplant, Test accuracy: 0.232 +/- 0.001, Weighted F1: 0.250 +/- 0.001
  Tested on batch 10x_Biopsy, Test accuracy: 0.566 +/- 0.002, Weighted F1: 0.532 +/- 0.004
  Tested on batch 10x_Transplant, Test accuracy: 0.695 +/- 0.004, Weighted F1: 0.566 +/- 0.004
Embedding: scgpt (fine-tuned) (mean +/- std over 3 seed(s))
  Tested on batch Dropseq_transplant, Test accuracy: 0.313 +/- 0.001, Weighted F1: 0.278 +/- 0.002
  Tested on batch 10x_Biopsy, Test accuracy: 0.507 +/- 0.000, Weighted F1: 0.478 +/- 0.002
  Tested on batch 10x_Transplant, Test accuracy: 0.71

In [26]:
# Build a transposed LaTeX table for all pancreas embedding methods.
metric_key = "macro_f1"  # Change to "macro_f1" if you want the Weighted F1 table.
metric_label = "Macro F1"

metric_by_method = {}
for method_name, batch_metrics in results.items():
    pretty_name = method_name
    metric_by_method[pretty_name] = pd.Series(
        {batch_name: metrics[metric_key] for batch_name, metrics in batch_metrics.items()}
    )

metric_df = pd.DataFrame(metric_by_method)
metric_df.index.name = "batch"
metric_df = metric_df.reset_index()
metric_df["batch"] = metric_df["batch"]
metric_df = metric_df.groupby("batch", as_index=True).mean(numeric_only=True).sort_index()

# Add deltas as steered - non-steered for each comparable pair.
delta_pairs = [
    ("scgpt (zero-shot, steered)", "scgpt (zero-shot)", "$\\Delta$ scgpt (zero-shot)"),
    ("scgpt (fine-tuned, steered)", "scgpt (fine-tuned)", "$\\Delta$ scgpt (fine-tuned)"),
    (
        "geneformer (fine-tuned, steered)",
        "geneformer (fine-tuned)",
        "$\\Delta$ geneformer (fine-tuned)",
    ),
]
for steered_name, base_name, delta_name in delta_pairs:
    if {steered_name, base_name}.issubset(metric_df.columns):
        metric_df[delta_name] = metric_df[steered_name] - metric_df[base_name]

mean_row = pd.DataFrame(metric_df.mean(numeric_only=True)).T
mean_row.index = ["Mean"]
latex_df = pd.concat([metric_df, mean_row]).T
pretty_batch_names = {
    "10x_Biopsy": "Biopsy (10x)",
    "10x_Transplant": "Transplant (10x)",
    "Dropseq_transplant": "Transplant (Drop-seq)",
    "mean": "Mean",
}

latex_df.columns = latex_df.columns.map(pretty_batch_names)
latex_table = latex_df.to_latex(
    index=True,
    float_format="%.2f",
    caption=(
        f"Transposed cross-batch KNN {metric_label} comparison across pancreas "
        "embedding methods (mean over 3 seeds)."
    ),
    label="tab:pancreas_embedding_methods_transposed",
)

#print(latex_table.replace("\\", "\\\\"))
print(latex_table)

\begin{table}
\caption{Transposed cross-batch KNN Macro F1 comparison across pancreas embedding methods (mean over 3 seeds).}
\label{tab:pancreas_embedding_methods_transposed}
\begin{tabular}{lrrrr}
\toprule
 & Biopsy (10x) & Transplant (10x) & Transplant (Drop-seq) & NaN \\
\midrule
scgpt (zero-shot) & 0.44 & 0.49 & 0.21 & 0.38 \\
scgpt (zero-shot, steered) & 0.53 & 0.57 & 0.25 & 0.45 \\
scgpt (fine-tuned) & 0.48 & 0.55 & 0.28 & 0.44 \\
scgpt (fine-tuned, steered) & 0.50 & 0.54 & 0.37 & 0.47 \\
$\Delta$ scgpt (zero-shot) & 0.09 & 0.08 & 0.04 & 0.07 \\
$\Delta$ scgpt (fine-tuned) & 0.02 & -0.01 & 0.09 & 0.03 \\
\bottomrule
\end{tabular}
\end{table}



In [23]:
lung_labels["dataset"].value_counts()

dataset
10x_Transplant        12725
10x_Biopsy            10000
Dropseq_transplant     9701
Name: count, dtype: int64